In [1]:
import requests
import datetime 
import os 
import json

ModuleNotFoundError: No module named 'requests'

In [ ]:
#saya mendeklarasikan variabel berisi API key di os demi keamanan agar API Key saya tidak dilihat publik
api = os.getenv("MAPS_API_KEY")
if api is None:
    raise ValueError("API key not found")

In [48]:
location_data = []

In [49]:

def get_place_coords(place_name, api_key):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        'address': place_name,
        'key': api_key
    }
    res = requests.get(url, params=params)
    data = res.json()

    if data['status'] == 'OK':
        location = data['results'][0]['geometry']['location']
        area = data['results'][0]['formatted_address']
        place_data = {
            'place_name': place_name,
            'latitude': location['lat'],
            'longitude': location['lng'],
            'area': area
        }
        if place_data not in location_data:
            location_data.append(place_data)
        else:
            None
            
        return place_data
    else:
        return None




In [56]:
get_place_coords('MCD SANUR',api)

{'place_name': 'MCD SANUR',
 'latitude': -8.6825529,
 'longitude': 115.2594186,
 'area': 'Jl. Bypass Ngurah Rai No.109, Sanur, Denpasar Selatan, Kota Denpasar, Bali 80228, Indonesia'}

In [ ]:
get_place_coords('PLAZA RENON',api)

{'place_name': 'PLAZA RENON',
 'latitude': -8.6732605,
 'longitude': 115.244127,
 'area': 'Jl. Raya Puputan No.210, Renon, Kec. Denpasar Tim., Kota Denpasar, Bali 80239, Indonesia'}

In [57]:
location_data

[{'place_name': 'MCD SANUR',
  'latitude': -8.6825529,
  'longitude': 115.2594186,
  'area': 'Jl. Bypass Ngurah Rai No.109, Sanur, Denpasar Selatan, Kota Denpasar, Bali 80228, Indonesia'}]

In [50]:
destination = []

In [117]:
#saya ingin download dan merapikan data json yang di crate API agar mudah saya baca
download_json = []

In [ ]:
def search_nearby_place(locationdata,radius_meters,type_place,api_key):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

    #latitude dan longitude centroid berdasarkan rata2 latitude dan longitude
    #untuk mencari titik tengah dari lokasi-lokasi yang kita masukkan,terinsiprasi dari k means clustering
    all_latitude = [place['latitude'] for place in locationdata]
    all_longitude = [place['longitude'] for place in locationdata]


    lat_centroid = sum(all_latitude)/len(all_latitude)
    lng_centroid = sum(all_longitude)/len(all_longitude)
    
    params = {
        "location": f"{lat_centroid},{lng_centroid}",
        "radius": radius_meters,
        "type": type_place,
        "key": api_key
    }
  

    res = requests.get(url, params=params)
    data = res.json()

    #ARSIP
    # return data
    
    #sebelum lanjut membuat code untuk menambahkan data ke variabel destination,saya akan coba download file json/var datanya
    #bertujuan agar saya dapat melihat struktur data agar mudah dirapikan dan dimasukan ke variable destination
    # download_json.append(data)
    
    #setelah download data dan melihat struktur data json,saya lanjut mengambil 3 parameter utama berdasar struktur json yang sudah saya baca
    place_data = [{
        "place_name":place['name'],
        "latitude": place['geometry']['location']['lat'],
        "longitude": place['geometry']['location']['lng'],
        "rating": place.get('rating',0)
    } for place in data['results']]
    
    if place_data not in destination:
        destination.extend(place_data)
    else:
        None
    
    return place_data
    
    

In [58]:
search_nearby_place(location_data,4000,'hospital',api)

[{'place_name': 'Global Health Center',
  'latitude': -8.6885798,
  'longitude': 115.2589453,
  'rating': 4.8},
 {'place_name': 'Bali Mandara Regional General Hospital',
  'latitude': -8.7035694,
  'longitude': 115.2487979,
  'rating': 4},
 {'place_name': 'Klinik Kanaka Medika',
  'latitude': -8.6830196,
  'longitude': 115.2594013,
  'rating': 4.7},
 {'place_name': 'Bali International Hospital (Main Lobby and MCU Entrance)',
  'latitude': -8.6796052,
  'longitude': 115.2606906,
  'rating': 4.1},
 {'place_name': 'Health Screening Bali International Hospital',
  'latitude': -8.679480500000002,
  'longitude': 115.2605972,
  'rating': 3},
 {'place_name': 'Bali International Hospital (Emergency Entrance)',
  'latitude': -8.6790429,
  'longitude': 115.2591727,
  'rating': 4.7},
 {'place_name': 'Bluecross Medika Internasional',
  'latitude': -8.6793688,
  'longitude': 115.2619578,
  'rating': 5},
 {'place_name': 'Doctor Surya Associates',
  'latitude': -8.6772119,
  'longitude': 115.2586608,


In [59]:
destination

[[{'place_name': 'Global Health Center',
   'latitude': -8.6885798,
   'longitude': 115.2589453,
   'rating': 4.8},
  {'place_name': 'Bali Mandara Regional General Hospital',
   'latitude': -8.7035694,
   'longitude': 115.2487979,
   'rating': 4},
  {'place_name': 'Klinik Kanaka Medika',
   'latitude': -8.6830196,
   'longitude': 115.2594013,
   'rating': 4.7},
  {'place_name': 'Bali International Hospital (Main Lobby and MCU Entrance)',
   'latitude': -8.6796052,
   'longitude': 115.2606906,
   'rating': 4.1},
  {'place_name': 'Health Screening Bali International Hospital',
   'latitude': -8.679480500000002,
   'longitude': 115.2605972,
   'rating': 3},
  {'place_name': 'Bali International Hospital (Emergency Entrance)',
   'latitude': -8.6790429,
   'longitude': 115.2591727,
   'rating': 4.7},
  {'place_name': 'Bluecross Medika Internasional',
   'latitude': -8.6793688,
   'longitude': 115.2619578,
   'rating': 5},
  {'place_name': 'Doctor Surya Associates',
   'latitude': -8.6772119

In [60]:
#tadi kan destination variable itu list dalam list krna saya loop pakai list comprehension,jadi saya flatten
destination_flatten = destination[0]

In [ ]:
#ARSIP

# with open("nearby_search.json","w") as f:
#         json.dump(download_json,f)
# download_json.clear()

In [61]:
destination_flatten

[{'place_name': 'Global Health Center',
  'latitude': -8.6885798,
  'longitude': 115.2589453,
  'rating': 4.8},
 {'place_name': 'Bali Mandara Regional General Hospital',
  'latitude': -8.7035694,
  'longitude': 115.2487979,
  'rating': 4},
 {'place_name': 'Klinik Kanaka Medika',
  'latitude': -8.6830196,
  'longitude': 115.2594013,
  'rating': 4.7},
 {'place_name': 'Bali International Hospital (Main Lobby and MCU Entrance)',
  'latitude': -8.6796052,
  'longitude': 115.2606906,
  'rating': 4.1},
 {'place_name': 'Health Screening Bali International Hospital',
  'latitude': -8.679480500000002,
  'longitude': 115.2605972,
  'rating': 3},
 {'place_name': 'Bali International Hospital (Emergency Entrance)',
  'latitude': -8.6790429,
  'longitude': 115.2591727,
  'rating': 4.7},
 {'place_name': 'Bluecross Medika Internasional',
  'latitude': -8.6793688,
  'longitude': 115.2619578,
  'rating': 5},
 {'place_name': 'Doctor Surya Associates',
  'latitude': -8.6772119,
  'longitude': 115.2586608,


In [37]:
travel_time= []

In [62]:

def get_travel_time(locationdata,destinationinput,departure_time,api_key):
    url = "https://maps.googleapis.com/maps/api/distancematrix/json"
    # for sublist in destination:
    #     destination_flattten.extend(sublist)
    #format untuk latitude longitude ke google api menggunakan |
    origin = "|".join([f"{o['latitude']},{o['longitude']}" for o in locationdata])
    destination_coor = "|".join([f"{d['latitude']},{d['longitude']}" for d in destinationinput])
    params = {
        "origins":origin,   
        "destinations":destination_coor,
        "mode":"driving",
        "departure_time":departure_time,
        "key":api_key
    }
    res = requests.get(url,params=params)
    data = res.json()

    #ARSIP
    
    #sebelum lanjut membuat code untuk menambahkan data ke variabel travel_time,saya akan coba download file json/var datanya
    #bertujuan agar saya dapat melihat struktur data agar mudah dirapikan dan dimasukan ke variable travel_time
    # download_json.append(data)

# merapikan variabel data,menam
    for i, row in enumerate(data['rows']):
        origin_name = locationdata[i]['place_name']
        for j,element in enumerate(row['elements']):
            destination_name = destinationinput[j]['place_name']
            alamat = data['destination_addresses'][j]
            durations= element['duration_in_traffic']['value']
            distance_in_meters = element['distance']['value']
            travel_time.append({
                "origin_name":origin_name,
                "destination_name":destination_name,
                "durations_in_seconds":durations,
                "distance_in_meters":distance_in_meters,
                "alamat":alamat
            })


In [63]:
get_travel_time(location_data,destination_flatten,'now',api)

In [ ]:
#Arsip
# with open("distance_matrix.json","w") as f:
#         json.dump(download_json,f)
# download_json.clear()

In [64]:
travel_time

[{'origin_name': 'MCD SANUR',
  'destination_name': 'Global Health Center',
  'durations_in_seconds': 140,
  'distance_in_meters': 726,
  'alamat': 'Jl. Bypass Ngurah Rai No.178, Sanur, Denpasar Selatan, Kota Denpasar, Bali 80228, Indonesia'},
 {'origin_name': 'MCD SANUR',
  'destination_name': 'Bali Mandara Regional General Hospital',
  'durations_in_seconds': 702,
  'distance_in_meters': 4673,
  'alamat': 'Jl. Bypass Ngurah Rai No.548, Sanur Kauh, Denpasar Selatan, Kota Denpasar, Bali, Indonesia'},
 {'origin_name': 'MCD SANUR',
  'destination_name': 'Klinik Kanaka Medika',
  'durations_in_seconds': 53,
  'distance_in_meters': 71,
  'alamat': '8785+QP7, Jl. Bypass Ngurah Rai No.113, Sanur, Denpasar Selatan, Kota Denpasar, Bali 80227, Indonesia'},
 {'origin_name': 'MCD SANUR',
  'destination_name': 'Bali International Hospital (Main Lobby and MCU Entrance)',
  'durations_in_seconds': 299,
  'distance_in_meters': 1367,
  'alamat': 'Jalan Kawasan Ekonomi Khusus Sanur, Lot H1-H2, Sanur Ka

In [ ]:

fastest_place = {}

# for data in travel_time:
#     name = data['origin_name']
#     distance_in_meters = data['distance_in_meters']
#     if name not in fastest_place or distance_in_meters < fastest_place[name]['distance_in_meters']:
#         fastest_place[name] = data
# fastest_place_arr = list(fastest_place.values())
    
for data in travel_time:
    name = data["origin_name"]
    durations_in_seconds = data["durations_in_seconds"]
    if (
        name not in fastest_place
        or durations_in_seconds < fastest_place[name]["durations_in_seconds"]
    ):
        fastest_place[name] = data
fastest_place_arr = list(fastest_place.values())

In [66]:
fastest_place_arr

[{'origin_name': 'MCD SANUR',
  'destination_name': 'Klinik Kanaka Medika',
  'durations_in_seconds': 53,
  'distance_in_meters': 71,
  'alamat': '8785+QP7, Jl. Bypass Ngurah Rai No.113, Sanur, Denpasar Selatan, Kota Denpasar, Bali 80227, Indonesia'}]

In [ ]:
def search_byrating(destinationinput):
    max_rating = 0
    top_places = []
    for i in destination_flatten:
        current_state = i.get("rating", 0)
        if current_state > max_rating:
            max_rating = current_state

    for x in destination_flatten:
        if x.get("rating", 0) == max_rating:
            top_places.append(x)

    print(f"Rating Tertinggi Ditemukan: {max_rating}")
    print("Daftar Tempat dengan Rating Tersebut:\n")
    for p in top_places:
        print(f"{p.get('place_name')} : rating {p.get('rating')}")


In [68]:
search_byrating(destination_flatten)

Rating Tertinggi Ditemukan: 5
Daftar Tempat dengan Rating Tersebut:

Bluecross Medika Internasional : rating 5
Doctor Surya Associates : rating 5
Rika Massage : rating 5
IMUTS Baby Spa RENON : rating 5
